In [1]:
import torch
from torch import nn
from transformers import PreTrainedModel, PretrainedConfig

# Step 0: From Assignment 1:

In [2]:
import torch, nltk, pickle
from torch import nn
from collections import Counter
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutput

from torch.utils.data import DataLoader
import numpy as np
import sys, time, os

from datasets import load_dataset
from types import SimpleNamespace

def lowercase_tokenizer(text):
    return [t.lower() for t in nltk.word_tokenize(text)]

import nltk
nltk.download('punkt_tab')
lowercase_tokenizer("Alpha is Beta")

def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None,
                    pad_token='<PAD>', unk_token='<UNK>', bos_token='<BOS>', eos_token='<EOS>'):
    """ Build a tokenizer from the given file.

        Args:
             train_file:        The name of the file containing the training texts.
             tokenize_fun:      The function that maps a text to a list of string tokens.
             max_voc_size:      The maximally allowed size of the vocabulary.
             model_max_length:  Truncate texts longer than this length.
             pad_token:         The dummy string corresponding to padding.
             unk_token:         The dummy string corresponding to out-of-vocabulary tokens.
             bos_token:         The dummy string corresponding to the beginning of the text.
             eos_token:         The dummy string corresponding to the end the text.
    """

    # TODO: build the vocabulary, possibly truncating it to max_voc_size if that is specified.
    # Then return a tokenizer object (implemented below).
    ...
    counter = Counter()

    with open(train_file, encoding='utf-8') as file:
        for paragraph in file:
            tokens = tokenize_fun(paragraph.strip())
            counter.update(tokens)

    words = []
    for word, count in counter.most_common():
        words.append(word)
    
    if max_voc_size is not None:
        words = words[:max_voc_size - 4]  # dummy tokens

    vocabulary = [pad_token, unk_token, bos_token, eos_token] + words

    return A1Tokenizer(
        vocabulary=vocabulary,
        tokenize_fun=tokenize_fun,
        model_max_length=model_max_length,
        pad_token=pad_token,
        unk_token=unk_token,
        bos_token=bos_token,
        eos_token=eos_token
    )

class A1Tokenizer:
    """A minimal implementation of a tokenizer similar to tokenizers in the HuggingFace library."""

    def __init__(self, vocabulary, tokenize_fun, model_max_length=None,
                 pad_token='<PAD>', unk_token='<UNK>',
                 bos_token='<BOS>', eos_token='<EOS>'):
        # TODO: store all values you need in order to implement __call__ below.
        self.vocabulary = vocabulary
        
        self.token2id = {}
        for i, tok in enumerate(vocabulary):
            self.token2id[tok] = i
            
        self.id2token = {}
        for tok, i in self.token2id.items():
            self.id2token[i] = tok

        self.tokenize_fun = tokenize_fun
        self.model_max_length = model_max_length

        self.pad_token = pad_token
        self.unk_token = unk_token
        self.bos_token = bos_token
        self.eos_token = eos_token

        self.pad_token_id = self.token2id[pad_token]
        self.unk_token_id = self.token2id[unk_token]
        self.bos_token_id = self.token2id[bos_token]
        self.eos_token_id = self.token2id[eos_token]

    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        """Tokenize the given texts and return a BatchEncoding containing the integer-encoded tokens.
           
           Args:
             texts:           The texts to tokenize.
             truncation:      Whether the texts should be truncated to model_max_length.
             padding:         Whether the tokenized texts should be padded on the right side.
             return_tensors:  If None, then return lists; if 'pt', then return PyTorch tensors.

           Returns:
             A BatchEncoding where the field `input_ids` stores the integer-encoded texts.
        """
        if return_tensors and return_tensors != 'pt':
            raise ValueError('Should be pt')
        
        # TODO: Your work here is to split the texts into words and map them to integer values.
        # 
        # - If `truncation` is set to True, the length of the encoded sequences should be 
        #   at most self.model_max_length.
        # - If `padding` is set to True, then all the integer-encoded sequences should be of the
        #   same length. That is: the shorter sequences should be "padded" by adding dummy padding
        #   tokens on the right side.
        # - If `return_tensors` is undefined, then the returned `input_ids` should be a list of lists.
        #   Otherwise, if `return_tensors` is 'pt', then `input_ids` should be a PyTorch 2D tensor.

        # TODO: Return a BatchEncoding where input_ids stores the result of the integer encoding.
        # Optionally, if you want to be 100% HuggingFace-compatible, you should also include an 
        # attention mask of the same shape as input_ids. In this mask, padding tokens correspond
        # to the the value 0 and real tokens to the value 1.

        encoded_sequences = []
        for text in texts:
            tokens = [self.bos_token] + self.tokenize_fun(text) + [self.eos_token]

            if truncation and self.model_max_length is not None:
                tokens = tokens[:self.model_max_length]
    
            ids = []
            for tok in tokens:
                if tok in self.token2id:
                    ids.append(self.token2id[tok])
                else:
                    ids.append(self.unk_token_id)

            encoded_sequences.append(ids)

        attention_mask = [[1] * len(seq) for seq in encoded_sequences]
        
        if padding:
            max_len = 0
            for seq in encoded_sequences:
                if len(seq) > max_len:
                    max_len = len(seq)

            padded_sequences = []
            padded_masks = []

            for seq, mask in zip(encoded_sequences, attention_mask):
                pad_len = max_len - len(seq)
                padded_sequences.append(seq + [self.pad_token_id] * pad_len)
                padded_masks.append(mask + [0] * pad_len)

            encoded_sequences = padded_sequences
            attention_mask = padded_masks

        if return_tensors == 'pt':
            encoded_sequences = torch.tensor(encoded_sequences, dtype=torch.long)
            attention_mask = torch.tensor(attention_mask, dtype=torch.long)

        return BatchEncoding({'input_ids': encoded_sequences, 'attention_mask': attention_mask})

    def __len__(self):
        """Return the size of the vocabulary."""
        return len(self.vocabulary)
    
    def save(self, filename):
        """Save the tokenizer to the given file."""
        with open(filename, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def from_file(filename):
        """Load a tokenizer from the given file."""
        with open(filename, 'rb') as f:
            return pickle.load(f)


max_voc_size = 10000

tokenizer = build_tokenizer(
    train_file="train.txt",
    max_voc_size=max_voc_size,
    model_max_length=128
)




TRAIN_FILE = "train.txt"
VAL_FILE = "val.txt"

dataset = load_dataset(
    "text",
    data_files={
        "train": TRAIN_FILE,
        "val": VAL_FILE
    }
)

dataset = dataset.filter(lambda x: x["text"].strip() != "")


class A1Trainer:
    """A minimal implementation similar to a Trainer from the HuggingFace library."""

    def __init__(self, model, args, train_dataset, eval_dataset, tokenizer):
        """Set up the trainer.
           
           Args:
             model:          The model to train.
             args:           The training parameters stored in a TrainingArguments object.
             train_dataset:  The dataset containing the training documents.
             eval_dataset:   The dataset containing the validation documents.
             eval_dataset:   The dataset containing the validation documents.
             tokenizer:      The tokenizer.
        """
        self.model = model
        self.args = args
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer

        assert(args.optim == 'adamw_torch')
        assert(args.eval_strategy == 'epoch')

    def select_device(self):
        """Return the device to use for training, depending on the training arguments and the available backends."""
        if self.args.use_cpu:
            return torch.device('cpu')
        if not self.args.no_cuda and torch.cuda.is_available():
            return torch.device('cuda')
        if torch.mps.is_available():
            return torch.device('mps')
        return torch.device('cpu')
            
    def train(self):
        """Train the model."""
        args = self.args

        device = self.select_device()
        print('Device:', device)
        self.model.to(device)
        
        loss_func = torch.nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id)

        # TODO: Relevant arguments: at least args.learning_rate, but you can optionally also consider
        # other Adam-related hyperparameters here.
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=args.learning_rate
        )

        # TODO: Relevant arguments: args.per_device_train_batch_size, args.per_device_eval_batch_size
        train_loader = DataLoader(
            self.train_dataset,
            batch_size=args.per_device_train_batch_size,
            shuffle=True
        )

        val_loader = DataLoader(
            self.eval_dataset,
            batch_size=args.per_device_eval_batch_size,
        )
        
        # TODO: Your work here is to implement the training loop.
        #       
        # for each training epoch (use args.num_train_epochs here):
        #   for each batch B in the training set:
        #
        #       PREPROCESSING AND FORWARD PASS:
        #       input_ids = apply your tokenizer to B
        #       labels = input_ids with padding replaced by -100
	    #       put input_ids and labels onto the GPU (or whatever device you use)
        #       apply the model to input_ids and labels
        #       get the loss from the model output
        #
        #       BACKWARD PASS AND MODEL UPDATE:
        #       optimizer.zero_grad()
        #       loss.backward()
        #       optimizer.step()

        loss_fct = torch.nn.CrossEntropyLoss(
            ignore_index=-100,
            reduction="sum"
        )

        for epoch in range(int(args.num_train_epochs)):
            self.model.train()
    
            total_train_loss = 0.0
            num_train_batches = 0
    
            for batch in train_loader:
                encoded = self.tokenizer(
                    batch["text"],
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                )
    
                input_ids = encoded["input_ids"].to(device)
    
                labels = input_ids.clone()
                labels[labels == self.tokenizer.pad_token_id] = -100
    
                output = self.model(
                    input_ids=input_ids,
                    labels=labels
                )
    
                loss = output.loss
    
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    
                total_train_loss += loss.item()
                num_train_batches += 1
    
            avg_train_loss = total_train_loss / num_train_batches
    

            self.model.eval()
    
            total_val_nll = 0.0
            total_val_tokens = 0
    
            with torch.no_grad():
                for batch in val_loader:
                    encoded = self.tokenizer(
                        batch["text"],
                        return_tensors="pt",
                        padding=True,
                        truncation=True
                    )
    
                    input_ids = encoded["input_ids"].to(device)
    
                    labels = input_ids.clone()
                    labels[labels == self.tokenizer.pad_token_id] = -100
    
                    output = self.model(input_ids=input_ids)
                    logits = output.logits
    
                    # For language modeling, prediction at position t
                    # should be compared with token at position t+1.
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = labels[:, 1:].contiguous()
    
                    val_loss_sum = loss_fct(
                        shift_logits.view(-1, shift_logits.size(-1)),
                        shift_labels.view(-1)
                    )
    
                    num_tokens = (shift_labels != -100).sum().item()
    
                    total_val_nll += val_loss_sum.item()
                    total_val_tokens += num_tokens
    
            avg_val_loss = total_val_nll / total_val_tokens
            perplexity = np.exp(avg_val_loss)
    
            print(
                f"Epoch {epoch + 1}: "
                f"train loss = {avg_train_loss:.4f}, "
                f"val loss = {avg_val_loss:.4f}, "
                f"perplexity = {perplexity:.2f}"
            )
    
        print(f"Saving to {args.output_dir}.")
    
        # Avoid save_pretrained error for config classes without default init values.
        if hasattr(self.model, "config"):
            self.model.config.has_no_defaults_at_init = True
    
        self.model.save_pretrained(args.output_dir)



def predict_next_words(model, tokenizer, text, k=5):
    """
    Predict the top-k next words after a given input text.
    """
    model.eval()
    device = next(model.parameters()).device

    encoded = tokenizer(
        [text],
        return_tensors="pt",
        padding=False,
        truncation=True
    )

    input_ids = encoded["input_ids"].to(device)

    with torch.no_grad():
        output = model(input_ids=input_ids)

    logits = output.logits

    # The tokenizer adds an EOS token at the end.
    # We want to predict the word after the last real input word,
    # so we use the second-last position.
    next_word_logits = logits[0, -2, :]

    scores, indices = torch.topk(next_word_logits, k=k)

    predictions = []

    for score, idx in zip(scores, indices):
        token_id = idx.item()
        token = tokenizer.id2token[token_id]
        predictions.append((token, score.item()))

    return predictions

def evaluate_perplexity(model, tokenizer, eval_dataset, batch_size=64):
    
    model.eval()
    device = next(model.parameters()).device

    loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    loss_fct = torch.nn.CrossEntropyLoss(
        ignore_index=-100,
        reduction="sum"
    )

    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            encoded = tokenizer(
                batch["text"],
                return_tensors="pt",
                padding=True,
                truncation=True
            )

            input_ids = encoded["input_ids"].to(device)

            labels = input_ids.clone()
            labels[labels == tokenizer.pad_token_id] = -100

            output = model(input_ids=input_ids)
            logits = output.logits

            # Predict token t+1 from position t.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

            num_tokens = (shift_labels != -100).sum().item()

            total_nll += loss.item()
            total_tokens += num_tokens

    mean_nll = total_nll / total_tokens
    perplexity = np.exp(mean_nll)

    return mean_nll, perplexity

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /cephyr/users/yingshuo/Alvis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Step 1: Setting up a Transformer neural network

In [3]:
class A2ModelConfig(PretrainedConfig):
    """Configuration object that stores hyperparameters that define the Transformer language model."""
    def __init__(self, vocab_size=None, hidden_size=None, intermediate_size=None, num_attention_heads=None, 
                 num_hidden_layers=None,
                 rope_theta=None, hidden_act='silu', max_position_embeddings=None, rms_norm_eps=None, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers

##  Task 1.1: MLP layer

In [4]:
class A2MLP(nn.Module):
    """The MLP layer of the Transformer. Uses the SwiGLU architecture."""
    def __init__(self, config):
        super().__init__()
        assert(config.hidden_act == 'silu')
        # TODO: initalize components here

        self.Linear1 = nn.Linear(config.hidden_size,config.intermediate_size, bias=False)
        self.Linear2 = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.Linear3 = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

        self.activation = nn.SiLU()

    def forward(self, hidden_states):
        gate = self.activation(self.Linear1(hidden_states))
        up = self.Linear2(hidden_states)

        hidden_states = gate * up
        hidden_states = self.Linear3(hidden_states)

        return hidden_states

### Sanity Check

In [5]:
config = A2ModelConfig(
    vocab_size=1000,
    hidden_size=16,
    intermediate_size=64,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    hidden_act='silu',
    max_position_embeddings=128,
    rms_norm_eps=1e-6
)

mlp = A2MLP(config)

x = torch.randn(2, 5, config.hidden_size)
y = mlp(x)

print(x.shape)
print(y.shape)

assert y.shape == x.shape
print("Sanity check 1.1: PASSED")

torch.Size([2, 5, 16])
torch.Size([2, 5, 16])
Sanity check 1.1: PASSED


## Task 1.2: Normalization

In [6]:
# This is optional, since you can use PyTorch's RMSNorm.
class A2RMSNorm(nn.Module):
    """RMS layer normalization."""
    def __init__(self, config):
        super().__init__()
        # TODO: Use config.rms_norm_eps
        # TODO: initalize weights here
        
        self.eps = config.rms_norm_eps
        self.weight = nn.Parameter(torch.ones(config.hidden_size))

    def forward(self, hidden_states):
        variance = hidden_states.pow(2).mean(dim=-1, keepdim=True)

        hidden_states = hidden_states / torch.sqrt(variance + self.eps)

        return self.weight * hidden_states

### Sanity Check

In [7]:
config = A2ModelConfig(
    vocab_size=1000,
    hidden_size=16,
    intermediate_size=64,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    hidden_act='silu',
    max_position_embeddings=128,
    rms_norm_eps=1e-6
)

norm = A2RMSNorm(config)

x = torch.randn(2, 5, config.hidden_size)
y = norm(x)

print(x.shape)
print(y.shape)

assert y.shape == x.shape
print("Sanity check 1.2: PASSED")

torch.Size([2, 5, 16])
torch.Size([2, 5, 16])
Sanity check 1.2: PASSED


## Task 1.3: Multi-head attention

In [8]:
#### RoPE implementation (copied and simplified from HuggingFace). ####

def apply_rotary_pos_emb(q, k, rope_rotations, unsqueeze_dim=1):
    """Applies precomputed RoPE rotations to the query and key representations."""
    assert(q.shape == k.shape)
    assert(len(q.shape) == 4)
    cos, sin = rope_rotations
    assert(q.shape[2] == cos.shape[1])
    assert(q.shape[3] == cos.shape[2])    
    q_type, k_type = q.dtype, k.dtype
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed.to(q_type), k_embed.to(k_type)

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

class A2RotaryEmbedding(nn.Module):
    """RoPE position representation for use in Transformer attention."""

    def __init__(self, config, device=None):
        super().__init__()
        rope_theta = config.rope_theta
        head_dim = config.hidden_size // config.num_attention_heads
        partial_rotary_factor = 1.0
        dim = int(head_dim * partial_rotary_factor)
        self.inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim))

    @torch.no_grad()
    def forward(self, x):
        position_ids = torch.arange(0, x.shape[1], device=x.device).unsqueeze(0)
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()

        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != "mps" else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):  # Force float32
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos()
            sin = emb.sin()
            return cos, sin

In [9]:
class A2Attention(nn.Module):
    """The multi-head attention layer of the Transformer. Uses standard scaled dot-product attention with causal masking."""
    
    def __init__(self, config):
        super().__init__()
        # TODO: set up W_q, W_k, W_v, W_o here
        # TODO: set up normalizers here

        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads

        self.W_q = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_k = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_v = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_o = nn.Linear(config.hidden_size, config.hidden_size, bias=False)

        self.q_norm = A2RMSNorm(config)
        self.k_norm = A2RMSNorm(config)

    def forward(self, hidden_states, rope_rotations):
        b, m, d = hidden_states.shape
        n_h = self.num_attention_heads
        d_h = self.head_dim

        q = self.W_q(hidden_states)
        k = self.W_k(hidden_states)
        v = self.W_v(hidden_states)

        q = self.q_norm(q)
        k = self.k_norm(k)

        q = q.view(b, m, n_h, d_h).transpose(1, 2)
        k = k.view(b, m, n_h, d_h).transpose(1, 2)
        v = v.view(b, m, n_h, d_h).transpose(1, 2)

        q, k = apply_rotary_pos_emb(q, k, rope_rotations)

        output = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)

        output = output.transpose(1, 2).reshape(b, m, d)

        output = self.W_o(output)

        return output

### Sanity Check

In [10]:
config = A2ModelConfig(
    vocab_size=1000,
    hidden_size=16,
    intermediate_size=64,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    hidden_act='silu',
    max_position_embeddings=128,
    rms_norm_eps=1e-6
)

attn = A2Attention(config)
rotary_emb = A2RotaryEmbedding(config)

input_ids = torch.randint(0, config.vocab_size, (2, 5))
x = torch.randn(2, 5, config.hidden_size)

rope_rotations = rotary_emb(input_ids)
y = attn(x, rope_rotations)

print(x.shape)
print(y.shape)

assert y.shape == x.shape
print("Sanity check 1.3: PASSED")

torch.Size([2, 5, 16])
torch.Size([2, 5, 16])
Sanity check 1.3: PASSED


## Task 1.4: The full Transformer decoder layer

In [11]:
class A2DecoderLayer(nn.Module):
    """A complete Transformer decoder layer."""
    def __init__(self, config):
        super().__init__()
        # TODO: set up attention, MLP, and normalizers here.

        self.self_attn = A2Attention(config)
        self.mlp = A2MLP(config)

        self.input_layernorm = A2RMSNorm(config)
        self.post_attention_layernorm = A2RMSNorm(config)


    def forward(self, hidden_states, rope_rotations):
        # RMSNorm -> Attention -> Residual add
        residual = hidden_states

        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, rope_rotations)

        hidden_states = residual + hidden_states

        # RMSNorm -> MLP -> Residual add
        residual = hidden_states

        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)

        hidden_states = residual + hidden_states

        return hidden_states

### Sanity Check

In [12]:
config = A2ModelConfig(
    vocab_size=1000,
    hidden_size=16,
    intermediate_size=64,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    hidden_act='silu',
    max_position_embeddings=128,
    rms_norm_eps=1e-6
)

layer = A2DecoderLayer(config)
rotary_emb = A2RotaryEmbedding(config)

input_ids = torch.randint(0, config.vocab_size, (2, 5))
x = torch.randn(2, 5, config.hidden_size)

rope_rotations = rotary_emb(input_ids)
y = layer(x, rope_rotations)

print(x.shape)
print(y.shape)

assert y.shape == x.shape
print("Sanity check 1.4: PASSED")

torch.Size([2, 5, 16])
torch.Size([2, 5, 16])
Sanity check 1.4: PASSED


## Task 1.5: The complete Transformer stack

In [13]:
class A2Transformer(PreTrainedModel):
    """A language model based on the Transformer architecture."""
    
    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)

        self.rotary_emb = A2RotaryEmbedding(config)
        # TODO: Set up the other components here.
        # TODO: put all transformer decoder layers in a ModuleList.

        self.embendding_layer = nn.Embedding(
            config.vocab_size,
            config.hidden_size
        )

        self.decoder_layers = nn.ModuleList([
            A2DecoderLayer(config)
            for _ in range(config.num_hidden_layers)
        ])

        self.normalization = A2RMSNorm(config)

        self.heads = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=False
        )

        # This line should be called after you have set up all components.
        self.post_init()


    def forward(self, input_ids, labels=None):
        rope_rotations = self.rotary_emb(input_ids) # pass this to all the transformer decoder layers

        # TODO: Call embedding, transformer decoder layers, last normalizer, and unembedding.
        # TODO: Compute the loss as in Assignment 1 if labels is not None.

        hidden_states = self.embendding_layer(input_ids)

        for layer in self.decoder_layers:
            hidden_states = layer(hidden_states, rope_rotations)

        hidden_states = self.normalization(hidden_states)

        logits = self.heads(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = torch.nn.functional.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits
        )

### Sanity Check

In [14]:
config = A2ModelConfig(
    vocab_size=1000,
    hidden_size=16,
    intermediate_size=64,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    hidden_act='silu',
    max_position_embeddings=128,
    rms_norm_eps=1e-6
)

model = A2Transformer(config)

input_ids = torch.randint(0, config.vocab_size, (2, 5))

output = model(input_ids)
logits = output.logits

print(input_ids.shape)
print(logits.shape)

assert logits.shape == (2, 5, config.vocab_size)
print("Sanity check 1.5.1: PASSED")

torch.Size([2, 5])
torch.Size([2, 5, 1000])
Sanity check 1.5.1: PASSED


In [15]:
labels = input_ids.clone()

output = model(input_ids, labels=labels)

loss = output.loss
logits = output.logits

print(loss)
print(logits.shape)

assert logits.shape == (2, 5, config.vocab_size)
assert loss.ndim == 0
print("Sanity check 1.5.2: PASSED")

tensor(6.8782, grad_fn=<NllLossBackward0>)
torch.Size([2, 5, 1000])
Sanity check 1.5.2: PASSED


# Step 2: Training the language model

In [16]:
import math
import torch
from torch.optim import AdamW

## Task 2.1: Training the language model

In [17]:
config = A2ModelConfig(
    vocab_size=len(tokenizer),
    hidden_size=128,
    intermediate_size=512,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    hidden_act="silu",
    max_position_embeddings=128,
    rms_norm_eps=1e-6
)

model = A2Transformer(config)

args = SimpleNamespace(
    output_dir="trainer_output_a2",
    optim="adamw_torch",
    eval_strategy="epoch",
    learning_rate=3e-4,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    use_cpu=False,
    no_cuda=False
)

trainer = A1Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    tokenizer=tokenizer
)

trainer.train()

Device: cuda
Epoch 1: train loss = 4.8340, val loss = 4.3564, perplexity = 77.97
Epoch 2: train loss = 4.2329, val loss = 4.1689, perplexity = 64.65
Epoch 3: train loss = 4.0648, val loss = 4.0978, perplexity = 60.21
Epoch 4: train loss = 3.9717, val loss = 4.0541, perplexity = 57.64
Epoch 5: train loss = 3.9110, val loss = 4.0311, perplexity = 56.32
Saving to trainer_output_a2.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Step 3: Generating text
## Task 3.1: Predicting the next word

In [18]:
# From Assignment 1

def predict_next_words(model, tokenizer, text, k=5):
    model.eval()
    device = next(model.parameters()).device

    encoded = tokenizer(
        [text],
        return_tensors="pt",
        padding=False,
        truncation=True
    )

    input_ids = encoded["input_ids"].to(device)

    with torch.no_grad():
        output = model(input_ids=input_ids)

    logits = output.logits
    next_word_logits = logits[0, -2, :]

    scores, indices = torch.topk(next_word_logits, k=k)

    predictions = []
    for score, idx in zip(scores, indices):
        token_id = idx.item()
        token = tokenizer.id2token[token_id]
        predictions.append((token, score.item()))

    return predictions

In [19]:
examples = [
    "She lives in San"
]

for text in examples:
    print("Input:", text)
    predictions = predict_next_words(model, tokenizer, text, k=5)

    for token, score in predictions:
        print(f"  {token:15s} {score:.4f}")

    print()

Input: She lives in San
  <UNK>           11.5161
  francisco       11.1487
  diego           10.6599
  antonio         8.8847
  juan            8.4010



## Task 3.2: Generating texts

In [20]:
from torch.distributions import Categorical

def generate_text(model, tokenizer, prompt, max_length=50, temperature=1.0, topk=10):
    
    model.eval()
    device = next(model.parameters()).device

    encoded = tokenizer(
        [prompt],
        return_tensors="pt",
        padding=False,
        truncation=True
    )

    input_ids = encoded["input_ids"].to(device)
    generated_ids = input_ids[0, :-1].tolist()
    eos_token_id = tokenizer.eos_token_id

    for _ in range(max_length):
        current_input = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():
            output = model(input_ids=current_input)

        next_token_logits = output.logits[0, -1, :]

        # temperature.
        next_token_logits = next_token_logits / temperature

        # Top-k
        topk_logits, topk_indices = torch.topk(next_token_logits, k=topk)

        distr = Categorical(logits=topk_logits)
        sampled_position = distr.sample()

        next_token_id = topk_indices[sampled_position].item()

        if next_token_id == eos_token_id:
            break

        generated_ids.append(next_token_id)

    if generated_ids[0] == tokenizer.bos_token_id:
        generated_ids = generated_ids[1:]

    tokens = [
        tokenizer.id2token[token_id]
        for token_id in generated_ids
        if token_id not in {
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
            tokenizer.pad_token_id
        }
    ]

    return " ".join(tokens)

In [21]:
prompts = [
    "In natural language processing, a Transformer",
    "Is Stockholm the capital of Sweden? Answer yes or no. The answer is",
    "Write a Python program that reverses a list."
]

for prompt in prompts:
    print("PROMPT:", prompt)
    print(generate_text(
        model,
        tokenizer,
        prompt,
        max_length=40,
        temperature=1.0,
        topk=10
    ))
    print()

PROMPT: In natural language processing, a Transformer
in natural language processing , a <UNK> method , is a <UNK> for <UNK> , and is defined as follows :

PROMPT: Is Stockholm the capital of Sweden? Answer yes or no. The answer is
is stockholm the capital of sweden ? answer yes or no . the answer is the <UNK> of a single , which is the most important city and a city , and the city of <UNK> ( <UNK> ) , which is the largest urban area of the state , is located in a <UNK>

PROMPT: Write a Python program that reverses a list.
write a <UNK> program that <UNK> a list . in contrast , the <UNK> program `` a '' is a <UNK> of <UNK> <UNK> <UNK> ' , which is also <UNK> ' for the <UNK> ' , <UNK> ' , <UNK> ' and <UNK> ' '' .



## Task 3.3: Comparing to a pre-trained Transformer

In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "allenai/OLMo-2-0425-1B"
olmo_tokenizer = AutoTokenizer.from_pretrained(model_name)
olmo_model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
olmo_model = olmo_model.to(device)
olmo_model.eval()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Olmo2ForCausalLM(
  (model): Olmo2Model(
    (embed_tokens): Embedding(100352, 2048, padding_idx=100277)
    (layers): ModuleList(
      (0-15): 16 x Olmo2DecoderLayer(
        (self_attn): Olmo2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Olmo2RMSNorm((2048,), eps=1e-06)
          (k_norm): Olmo2RMSNorm((2048,), eps=1e-06)
        )
        (mlp): Olmo2MLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Olmo2RMSNorm((2048,), eps=1e-06

In [24]:
def generate_with_hf_model(model, tokenizer, prompt, max_new_tokens=50):
    device = next(model.parameters()).device

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [25]:
prompts = [
    "In natural language processing, a Transformer",
    "Is Stockholm the capital of Sweden? Answer yes or no. The answer is",
    "Write a Python program that reverses a list."
]

for prompt in prompts:
    print("=" * 80)
    print("PROMPT:", prompt)
    print()
    print(generate_with_hf_model(
        olmo_model,
        olmo_tokenizer,
        prompt,
        max_new_tokens=80
    ))
    print()

PROMPT: In natural language processing, a Transformer

In natural language processing, a Transformer is a type of neural network architecture that is used for natural language processing tasks. It was developed by Google Brain researchers in 2017 and has since become one of the most popular architectures for NLP tasks. The Transformer is a self-attention mechanism that allows the model to learn from the context of the input sentence. This makes it particularly well-suited for tasks like machine translation, where the goal

PROMPT: Is Stockholm the capital of Sweden? Answer yes or no. The answer is

Is Stockholm the capital of Sweden? Answer yes or no. The answer is yes.

PROMPT: Write a Python program that reverses a list.

Write a Python program that reverses a list. The list should be reversed in place, so that the original list is unchanged. For example, given the list [1, 2, 3], the output should be [3, 2, 1]. The list should be reversed in place, so that the original list is uncha